# Ranking and Relevance

## Overview
FTS returns matching documents — but in what order? Ranking assigns a relevance score to each result so the most useful documents appear first. PostgreSQL provides `ts_rank()` (frequency-based) and `ts_rank_cd()` (cover density — rewards terms that appear close together).

**Ranking functions:**
```sql
ts_rank(tsvector, tsquery)              -- raw frequency score
ts_rank(tsvector, tsquery, normalization) -- with length normalisation
ts_rank_cd(tsvector, tsquery)           -- cover density (proximity-aware)
```

**setweight() for field importance:**
```sql
-- Title matches rank higher than body text:
setweight(to_tsvector('english', title),    'A') ||   -- weight A = highest
setweight(to_tsvector('english', abstract), 'B') ||   -- weight B
setweight(to_tsvector('english', keywords), 'A')      -- weight A
```

---

In [1]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Clinical notes and patient records for FTS demos
CREATE TABLE patients (
    patient_id  TEXT PRIMARY KEY,
    full_name   TEXT NOT NULL,
    province    TEXT NOT NULL,
    dob         TEXT NOT NULL
);

CREATE TABLE clinical_notes (
    note_id     TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL REFERENCES patients(patient_id),
    note_date   TEXT NOT NULL,
    provider    TEXT NOT NULL,
    note_type   TEXT NOT NULL,  -- 'progress','discharge','consult','operative'
    content     TEXT NOT NULL
);

CREATE TABLE medications (
    med_id      TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL,
    drug_name   TEXT NOT NULL,
    indication  TEXT,
    notes       TEXT
);

CREATE TABLE research_articles (
    article_id  TEXT PRIMARY KEY,
    title       TEXT NOT NULL,
    abstract    TEXT NOT NULL,
    keywords    TEXT,
    published   TEXT NOT NULL
);

-- FTS5 virtual tables (SQLite equivalent of PostgreSQL tsvector index)
CREATE VIRTUAL TABLE notes_fts USING fts5(
    note_id UNINDEXED,
    patient_id UNINDEXED,
    note_type UNINDEXED,
    content,
    tokenize='porter ascii'   -- porter stemmer (closest to pg's english dictionary)
);

CREATE VIRTUAL TABLE articles_fts USING fts5(
    article_id UNINDEXED,
    title,
    abstract,
    keywords,
    tokenize='porter ascii'
);
""")

patients = [
    ('P001','Alice Ngata',      'NB','1980-03-15'),
    ('P002','Bob Chen',         'ON','1972-07-22'),
    ('P003','Fatima Rashid',    'BC','1986-11-05'),
    ('P004','James MacLeod',    'NS','1963-01-30'),
    ('P005','Mei-Ling Tran',    'QC','1966-08-19'),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?)", patients)

notes = [
    ('N001','P001','2023-10-01','Dr. Lee','progress',
     'Patient presents with poorly controlled hypertension. Blood pressure 148/92. '
     'Currently on Lisinopril 10mg once daily. Discussed medication adherence and sodium restriction. '
     'Patient reports occasional dizziness and dry cough, a known side effect of ACE inhibitors. '
     'Recommended dietary changes and increased physical activity. Follow-up in 4 weeks.'),
    ('N002','P001','2024-01-15','Dr. Lee','progress',
     'Follow-up for hypertension management. Blood pressure improved to 132/84. '
     'Patient adherent to Lisinopril. Dry cough persists. Considering switching to ARB if cough continues. '
     'HbA1c 7.2%, borderline elevated. Discussed diabetes prevention strategies. '
     'Lipid panel ordered. Continue current antihypertensive therapy.'),
    ('N003','P002','2024-01-10','Dr. Pham','discharge',
     'Patient admitted for acute exacerbation of Type 2 Diabetes. HbA1c 8.4% on admission. '
     'Adjusted Metformin to 1000mg BID and added Empagliflozin 10mg daily. '
     'Nutritional counselling provided. Patient educated on carbohydrate counting and glucose monitoring. '
     'Discharged in stable condition with follow-up appointment in 2 weeks.'),
    ('N004','P003','2023-08-20','Dr. Singh','consult',
     'Respiratory consultation for persistent asthma exacerbations. '
     'Patient reports frequent nocturnal wheezing and dyspnoea on exertion. '
     'Peak flow measurements show significant variability. Spirometry confirms obstructive pattern. '
     'Current inhaler technique reviewed and corrected. Prescribed Fluticasone-Salmeterol combination inhaler. '
     'Referral for pulmonary rehabilitation. Advised to avoid known allergens including dust mites and pet dander.'),
    ('N005','P004','2023-09-01','Dr. Lee','progress',
     'Routine diabetic review. HbA1c 7.8%, improved from last visit. '
     'eGFR 72 mL/min, stable kidney function. Patient reports no symptoms of hypoglycaemia. '
     'Foot examination normal. Retinal screening up to date. '
     'Continue current diabetes management. Annual flu vaccination administered. Blood pressure well controlled.'),
    ('N006','P005','2024-02-01','Dr. Pham','progress',
     'Complex case: Type 2 Diabetes with Hypertension and CKD Stage 3. '
     'HbA1c 9.1%, suboptimal glycaemic control. eGFR 38 mL/min, declining renal function. '
     'Metformin contraindicated due to renal impairment. Switched to Insulin glargine 20 units nocte. '
     'Potassium 5.8 mmol/L, hyperkalaemia noted. Dietary potassium restriction advised. '
     'Nephrology referral placed. Strict blood pressure control essential to slow CKD progression.'),
    ('N007','P002','2023-05-15','Dr. Pham','operative',
     'Pre-operative assessment for elective cholecystectomy. '
     'Medical history: Type 2 Diabetes, Hypertension, well controlled on Metformin and Lisinopril. '
     'Electrocardiogram normal. Chest X-ray clear. Blood glucose within acceptable range. '
     'Patient advised to withhold Metformin 48 hours prior to surgery due to contrast risk. '
     'Anaesthesia consultation arranged. Patient consented and cleared for surgery.'),
    ('N008','P001','2024-03-15','Dr. Singh','consult',
     'Cardiology consult for management of resistant hypertension. '
     'Despite optimal doses of Lisinopril and Amlodipine, blood pressure remains elevated at 158/96. '
     'Echocardiogram shows mild left ventricular hypertrophy. '
     'Added Spironolactone 25mg daily for additional blood pressure reduction. '
     'Stress test recommended to rule out ischaemic heart disease. '
     'Patient to monitor blood pressure at home twice daily and maintain log.'),
]
conn.executemany("INSERT INTO clinical_notes VALUES (?,?,?,?,?,?)", notes)

articles = [
    ('A001','Management of Type 2 Diabetes with Renal Impairment',
     'Type 2 diabetes mellitus complicated by chronic kidney disease requires careful medication selection. '
     'Metformin should be avoided when eGFR falls below 30 mL/min due to risk of lactic acidosis. '
     'SGLT2 inhibitors demonstrate renoprotective effects and cardiovascular benefits. '
     'Insulin therapy remains an option at all stages of renal impairment.',
     'diabetes,CKD,renal impairment,Metformin,SGLT2 inhibitor','2023-06-01'),
    ('A002','Hypertension Treatment Guidelines: ACE Inhibitors and ARBs',
     'Angiotensin-converting enzyme inhibitors and angiotensin receptor blockers are first-line '
     'antihypertensive agents. ACE inhibitors commonly cause dry cough due to bradykinin accumulation. '
     'ARBs are preferred in patients who cannot tolerate ACE inhibitor cough. '
     'Both drug classes provide renoprotection in diabetic nephropathy.',
     'hypertension,ACE inhibitor,ARB,Lisinopril,blood pressure','2022-11-15'),
    ('A003','Asthma Exacerbation Management in Primary Care',
     'Acute asthma exacerbations require rapid assessment of severity using peak expiratory flow and '
     'oxygen saturation. Short-acting beta-agonists remain the cornerstone of acute bronchodilation. '
     'Inhaled corticosteroids reduce airway inflammation and prevent future exacerbations. '
     'Patients with frequent exacerbations should be referred for specialist pulmonary assessment.',
     'asthma,exacerbation,bronchodilator,corticosteroid,peak flow','2023-03-20'),
    ('A004','Glycaemic Control Targets in Hospitalised Patients',
     'Inpatient hyperglycaemia is associated with increased morbidity, infection risk, and length of stay. '
     'Target blood glucose range of 6-10 mmol/L is recommended for most hospitalised patients. '
     'Insulin infusion protocols allow precise glycaemic management in critically ill patients. '
     'HbA1c measurement on admission provides information about pre-admission glycaemic control.',
     'glycaemic control,HbA1c,insulin,hospitalisation,hyperglycaemia','2024-01-08'),
    ('A005','Chronic Kidney Disease Progression and Blood Pressure Control',
     'Hypertension is both a cause and consequence of chronic kidney disease. '
     'Strict blood pressure control below 130/80 mmHg slows CKD progression significantly. '
     'Renin-angiotensin-aldosterone system blockade with ACE inhibitors or ARBs is recommended '
     'for patients with proteinuria. Regular monitoring of eGFR and potassium is essential.',
     'CKD,hypertension,blood pressure,ACE inhibitor,eGFR,proteinuria','2023-09-12'),
]
conn.executemany("INSERT INTO research_articles VALUES (?,?,?,?,?)", articles)

meds = [
    ('M001','P001','Lisinopril',   'Hypertension','10mg once daily'),
    ('M002','P001','Amlodipine',   'Hypertension','5mg once daily'),
    ('M003','P002','Metformin',    'Type 2 Diabetes','500mg BID'),
    ('M004','P002','Lisinopril',   'Hypertension','5mg once daily'),
    ('M005','P003','Salbutamol',   'Asthma','PRN inhaler'),
    ('M006','P005','Insulin glargine','Type 2 Diabetes','20 units nocte'),
    ('M007','P004','Metformin',    'Type 2 Diabetes','500mg daily'),
]
conn.executemany("INSERT INTO medications VALUES (?,?,?,?,?)", meds)

# Populate FTS5 indexes
conn.execute("INSERT INTO notes_fts SELECT note_id,patient_id,note_type,content FROM clinical_notes")
conn.execute("""
    INSERT INTO articles_fts
    SELECT article_id, title, abstract, COALESCE(keywords,'') FROM research_articles
""")
conn.commit()

print("FTS healthcare dataset loaded:")
for tbl in ['patients','clinical_notes','medications','research_articles']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<22s}: {n} rows")
print("  notes_fts (FTS5)      : indexed")
print("  articles_fts (FTS5)   : indexed")


FTS healthcare dataset loaded:
  patients              : 5 rows
  clinical_notes        : 8 rows
  medications           : 7 rows
  research_articles     : 5 rows
  notes_fts (FTS5)      : indexed
  articles_fts (FTS5)   : indexed


---
## ts_rank: scoring relevance

In [2]:
print("=== ts_rank: scoring relevance ===")
print()

print("PostgreSQL ts_rank() and ts_rank_cd():")
print("""
  ts_rank(tsvector, tsquery)              -- frequency-based rank
  ts_rank(tsvector, tsquery, normalization) -- with length normalization
  ts_rank_cd(tsvector, tsquery)           -- cover density (proximity-aware)
""")

norm_table = [
    (0,  "Raw frequency (default) — longer documents score higher"),
    (1,  "Divide by 1 + log(document length)"),
    (2,  "Divide by document length"),
    (4,  "Divide by mean harmonic distance between extents"),
    (8,  "Divide by number of unique words"),
    (16, "Divide by 1 + log(number of unique words)"),
    (32, "Divide by rank itself (self-normalizing)"),
]
print("Normalization options (can be OR'd together, e.g. 1|2):")
print(f"  {'value':<6s}  Effect")
print("  " + "-"*55)
for val, desc in norm_table:
    print(f"  {val:<6d}  {desc}")

print()
# SQLite FTS5 BM25 ranking (closest to ts_rank_cd)
rows = conn.execute("""
    SELECT n.note_id,
           n.patient_id,
           n.note_type,
           ROUND(-notes_fts.rank, 4)  AS bm25_score,
           SUBSTR(n.content, 1, 80)   AS snippet
    FROM   notes_fts
    JOIN   clinical_notes n ON n.note_id = notes_fts.note_id
    WHERE  notes_fts MATCH 'diabetes'
    ORDER  BY notes_fts.rank          -- lower rank = more relevant in FTS5
""").fetchall()

print("Ranked search for 'diabetes' (BM25 — higher score = more relevant):")
print(f"  {'note_id':<8s}  {'patient':<8s}  {'type':<10s}  {'score':>8s}  snippet")
print("  " + "-"*75)
for r in rows:
    print(f"  {r['note_id']:<8s}  {r['patient_id']:<8s}  {r['note_type']:<10s}  "
          f"{r['bm25_score']:>8.4f}  {r['snippet'][:45]}...")

print()
print("PostgreSQL ts_rank with normalization:")
print("""
  SELECT note_id,
         ts_rank(to_tsvector('english', content),
                 to_tsquery('english', 'diabetes'), 1) AS rank
  FROM   clinical_notes
  WHERE  to_tsvector('english', content)
         @@ to_tsquery('english', 'diabetes')
  ORDER  BY rank DESC;

  -- ts_rank_cd (cover density — rewards terms close together):
  SELECT note_id,
         ts_rank_cd(
             to_tsvector('english', content),
             to_tsquery('english', 'blood & pressure'),
             32     -- self-normalizing
         ) AS rank_cd
  FROM   clinical_notes
  WHERE  to_tsvector('english', content)
         @@ to_tsquery('english', 'blood & pressure')
  ORDER  BY rank_cd DESC;
""")


=== ts_rank: scoring relevance ===

PostgreSQL ts_rank() and ts_rank_cd():

  ts_rank(tsvector, tsquery)              -- frequency-based rank
  ts_rank(tsvector, tsquery, normalization) -- with length normalization
  ts_rank_cd(tsvector, tsquery)           -- cover density (proximity-aware)

Normalization options (can be OR'd together, e.g. 1|2):
  value   Effect
  -------------------------------------------------------
  0       Raw frequency (default) — longer documents score higher
  1       Divide by 1 + log(document length)
  2       Divide by document length
  4       Divide by mean harmonic distance between extents
  8       Divide by number of unique words
  16      Divide by 1 + log(number of unique words)
  32      Divide by rank itself (self-normalizing)

Ranked search for 'diabetes' (BM25 — higher score = more relevant):
  note_id   patient   type           score  snippet
  ---------------------------------------------------------------------------
  N005      P004      pro

---
## ts_headline: highlighted snippets

In [3]:
print("=== ts_headline: search result snippets with highlighting ===")
print()

print("ts_headline() extracts a relevant fragment and marks matched terms.")
print()

print("PostgreSQL ts_headline() syntax:")
print("""
  SELECT note_id,
         ts_headline(
             'english',
             content,
             to_tsquery('english', 'hypertension & cough'),
             'MaxWords=30, MinWords=15, StartSel=<b>, StopSel=</b>'
         ) AS headline
  FROM   clinical_notes
  WHERE  to_tsvector('english', content)
         @@ to_tsquery('english', 'hypertension & cough');
""")

print("ts_headline options:")
headline_opts = [
    ("MaxWords=N",       "Maximum words in the headline fragment (default 35)"),
    ("MinWords=N",       "Minimum words in the headline fragment (default 15)"),
    ("ShortWord=N",      "Words shorter than N are never the start of a fragment (default 3)"),
    ("HighlightAll=true","Highlight all occurrences in full document (no fragment extraction)"),
    ("MaxFragments=N",   "Return N fragments separated by delimiter (default 0 = one fragment)"),
    ("FragmentDelimiter","String between multiple fragments (default '...')"),
    ("StartSel=<b>",     "HTML tag inserted before matched term"),
    ("StopSel=</b>",     "HTML tag inserted after matched term"),
]
print(f"  {'Option':<22s}  Description")
print("  " + "-"*60)
for opt, desc in headline_opts:
    print(f"  {opt:<22s}  {desc}")

print()
# SQLite FTS5 snippet() equivalent
rows = conn.execute("""
    SELECT n.note_id,
           snippet(notes_fts, 3, '[', ']', '...', 20) AS snip
    FROM   notes_fts
    JOIN   clinical_notes n ON n.note_id = notes_fts.note_id
    WHERE  notes_fts MATCH 'hypertension'
    ORDER  BY notes_fts.rank
    LIMIT  4
""").fetchall()

print("SQLite FTS5 snippet() for 'hypertension' ([ ] marks matched terms):")
for r in rows:
    print(f"  {r['note_id']}: {r['snip']}")
    print()


=== ts_headline: search result snippets with highlighting ===

ts_headline() extracts a relevant fragment and marks matched terms.

PostgreSQL ts_headline() syntax:

  SELECT note_id,
         ts_headline(
             'english',
             content,
             to_tsquery('english', 'hypertension & cough'),
             'MaxWords=30, MinWords=15, StartSel=<b>, StopSel=</b>'
         ) AS headline
  FROM   clinical_notes
  WHERE  to_tsvector('english', content)
         @@ to_tsquery('english', 'hypertension & cough');

ts_headline options:
  Option                  Description
  ------------------------------------------------------------
  MaxWords=N              Maximum words in the headline fragment (default 35)
  MinWords=N              Minimum words in the headline fragment (default 15)
  ShortWord=N             Words shorter than N are never the start of a fragment (default 3)
  HighlightAll=true       Highlight all occurrences in full document (no fragment extraction)
  MaxFr

---
## setweight: field weighting

In [5]:
print("=== Field weighting: title vs body ===")
print()

print("Documents often have multiple fields with different importance.")
print("Searches against titles should rank higher than matches in body text.")
print()

print("PostgreSQL setweight() for field weighting:")
print("""
  -- Weights: A (highest) > B > C > D (lowest)
  SELECT article_id,
         ts_rank(
             setweight(to_tsvector('english', title),    'A') ||
             setweight(to_tsvector('english', abstract), 'B') ||
             setweight(to_tsvector('english', keywords), 'A'),
             to_tsquery('english', 'diabetes')
         ) AS weighted_rank
  FROM   research_articles
  WHERE (setweight(to_tsvector('english', title),    'A') ||
         setweight(to_tsvector('english', abstract), 'B') ||
         setweight(to_tsvector('english', keywords), 'A'))
         @@ to_tsquery('english', 'diabetes')
  ORDER  BY weighted_rank DESC;
""")

# SQLite FTS5 equivalent with column weights
rows = conn.execute("""
    SELECT r.article_id, r.title,
           ROUND(-articles_fts.rank, 4) AS relevance
    FROM   articles_fts
    JOIN   research_articles r ON r.article_id = articles_fts.article_id
    WHERE  articles_fts MATCH 'title : diabetes OR abstract : diabetes'
    ORDER  BY articles_fts.rank
""").fetchall()

print("FTS5 column-weighted search for 'diabetes' across title + abstract:")
print(f"  {'article_id':<10s}  {'relevance':>10s}  title")
print("  " + "-"*65)
for r in rows:
    print(f"  {r['article_id']:<10s}  {r['relevance']:>10.4f}  {r['title'][:50]}")

print()
print("PostgreSQL stored tsvector with weights (efficient approach):")
print("""
  -- One stored tsvector combining all fields with weights:
  ALTER TABLE research_articles
      ADD COLUMN tsv tsvector
      GENERATED ALWAYS AS (
          setweight(to_tsvector('english', COALESCE(title,'')),    'A') ||
          setweight(to_tsvector('english', COALESCE(abstract,'')), 'B') ||
          setweight(to_tsvector('english', COALESCE(keywords,'')), 'A')
      ) STORED;

  CREATE INDEX idx_articles_tsv ON research_articles USING GIN (tsv);

  -- Query is now clean and fast:
  SELECT article_id, ts_rank(tsv, query) AS rank
  FROM   research_articles,
         to_tsquery('english', 'diabetes') query
  WHERE  tsv @@ query
  ORDER  BY rank DESC;
""")

=== Field weighting: title vs body ===

Documents often have multiple fields with different importance.
Searches against titles should rank higher than matches in body text.

PostgreSQL setweight() for field weighting:

  -- Weights: A (highest) > B > C > D (lowest)
  SELECT article_id,
         ts_rank(
             setweight(to_tsvector('english', title),    'A') ||
             setweight(to_tsvector('english', abstract), 'B') ||
             setweight(to_tsvector('english', keywords), 'A'),
             to_tsquery('english', 'diabetes')
         ) AS weighted_rank
  FROM   research_articles
  WHERE (setweight(to_tsvector('english', title),    'A') ||
         setweight(to_tsvector('english', abstract), 'B') ||
         setweight(to_tsvector('english', keywords), 'A'))
         @@ to_tsquery('english', 'diabetes')
  ORDER  BY weighted_rank DESC;

FTS5 column-weighted search for 'diabetes' across title + abstract:
  article_id   relevance  title
  -------------------------------------

---
## Common Pitfalls

**1. Using ts_rank without normalization on documents of varying length**
A 10,000-word discharge summary with 'diabetes' mentioned 5 times will outrank a 200-word progress note where 'diabetes' is the entire topic — purely because of length. Use normalization: `ts_rank(tsv, query, 1)` (divide by 1 + log(length)) or `ts_rank(tsv, query, 32)` (self-normalizing) for fairer ranking across documents of different lengths.

**2. Recomputing ts_rank for every row in a large table**
`ORDER BY ts_rank(to_tsvector('english', content), query) DESC` on a 1M-row table computes the rank for every matched row before sorting. For large result sets, use `LIMIT` early and consider materializing the tsvector column so ts_rank operates on pre-computed vectors.

**3. ts_rank_cd not improving results for single-term queries**
ts_rank_cd (cover density) improves ranking when multiple terms appear — it rewards documents where terms are close together. For single-term queries, ts_rank_cd and ts_rank produce identical results. Use ts_rank_cd only for multi-term queries.

**4. Ignoring setweight() for structured documents**
Clinical notes have structure: a match in `note_type = 'discharge'` is more significant than a match buried in a long narrative. Not using setweight() treats all text equally and produces weaker rankings. Always weight title, type, and keyword fields higher than body content.

**5. ts_headline generating misleading fragments**
ts_headline selects fragments based on match density. For very long notes, the selected fragment may lack clinical context (e.g., just 'diabetes' mid-sentence). Set `MaxFragments=2` or `MaxWords=50` to give users enough context to evaluate whether the result is actually relevant.

**6. Ranking without filtering — slow and irrelevant**
Running `ORDER BY ts_rank(...) DESC` without a `WHERE tsv @@ query` filter computes a rank score for every row in the table, including documents that don't match at all (rank=0). Always filter first with `WHERE tsv @@ query`, then rank only the matching rows.


---
*sql_methods_library - Samantha McGarrigle*